# Week 6 Interim Submission — Baseline Triage Model

**Student:** Vishal Baboolal  
**Goal:** Train and evaluate an initial logistic-regression baseline for predicting ESI triage level.

This notebook follows the Week 6 tutorial workflow:

1. Load the cleaned Week 5 dataset.
2. Remove target leakage and excluded fields.
3. Create an 80/20 stratified split.
4. Train a stratified random baseline.
5. Scale the training features and train logistic regression.
6. Evaluate the model with accuracy, per-class metrics, and a confusion matrix.
7. Interpret the result clinically, with emphasis on ESI Level 1.


In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    recall_score,
)

RANDOM_STATE = 42
print("Libraries loaded.")


Libraries loaded.


## 1. Load the cleaned Week 5 dataset

The raw patient-level dataset is not committed to the public repository. Place the cleaned CSV in Google Colab, Google Drive, or a local `data/` folder before running the notebook.


In [2]:
candidate_paths = [
    Path("data_cleaned_week5.csv"),
    Path("../data/data_cleaned_week5.csv"),
    Path("/content/data_cleaned_week5.csv"),
    Path("/content/drive/MyDrive/data_cleaned_week5.csv"),
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Upload data_cleaned_week5.csv or update DATA_PATH to its location."
    )

df = pd.read_csv(DATA_PATH)

print(f"Loaded {df.shape[0]:,} patient visits and {df.shape[1]} columns.")
print("Target distribution:")
print(df["esi"].value_counts().sort_index())


Loaded 55,121 patient visits and 225 columns.
Target distribution:
esi
1       77
2    17924
3    27010
4     8896
5     1214


## 2. Choose the features and target

`esi` is the prediction target. `disposition` and `previousdispo` are excluded because they are known after triage and would create target leakage. Administrative variables and fairness-sensitive demographic variables are excluded from this first clinical baseline.


In [3]:
TARGET = "esi"

DEMOGRAPHICS = [
    "age", "gender", "ethnicity", "race", "lang", "religion",
    "maritalstatus", "employstatus", "insurance_status"
]

ADMIN = [
    "dep_name", "arrivalmode", "arrivalmonth",
    "arrivalday", "arrivalhour_bin"
]

LEAKAGE = ["disposition", "previousdispo"]

FEATURES = [
    c for c in df.columns
    if c != TARGET and c not in LEAKAGE + ADMIN + DEMOGRAPHICS
]

X = df[FEATURES]
y = df[TARGET]

print(f"Model input features: {len(FEATURES)}")
print(f"Missing values in model matrix: {int(X.isna().sum().sum())}")
print(f"All model features numeric: {all(pd.api.types.is_numeric_dtype(X[c]) for c in X.columns)}")


Model input features: 208
Missing values in model matrix: 0
All model features numeric: True


## 3. Create an 80/20 stratified train/test split

Stratification keeps the same ESI class proportions in both sets. The fixed random seed makes the results reproducible.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Training records: {len(X_train):,}")
print(f"Testing records:  {len(X_test):,}")


Training records: 44,096
Testing records:  11,025


## 4. Build a stratified random baseline

This deliberately simple classifier guesses labels in approximately the same proportions as the training data. A useful model should outperform it.


In [5]:
dummy = DummyClassifier(
    strategy="stratified",
    random_state=RANDOM_STATE,
)
dummy.fit(X_train, y_train)
pred_dummy = dummy.predict(X_test)

dummy_accuracy = accuracy_score(y_test, pred_dummy)
print(f"Stratified random baseline accuracy: {dummy_accuracy:.3f}")


Stratified random baseline accuracy: 0.375


## 5. Train the logistic-regression baseline

The scaler is fitted only on the training set, then applied to the held-out test set. This prevents information from the test set leaking into training.


In [6]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
)
logreg.fit(X_train_scaled, y_train)

pred_logreg = logreg.predict(X_test_scaled)

logreg_accuracy = accuracy_score(y_test, pred_logreg)
print(f"Logistic-regression accuracy: {logreg_accuracy:.3f}")


Logistic-regression accuracy: 0.667


## 6. Initial evaluation metrics

Accuracy alone can hide poor performance on rare classes. The full classification report shows precision, recall, and F1 for every ESI level.


In [7]:
print(classification_report(
    y_test,
    pred_logreg,
    labels=[1, 2, 3, 4, 5],
    digits=3,
    zero_division=0,
))

macro_f1 = f1_score(y_test, pred_logreg, average="macro")
weighted_f1 = f1_score(y_test, pred_logreg, average="weighted")
esi1_recall = recall_score(
    y_test,
    pred_logreg,
    labels=[1],
    average=None,
)[0]

print(f"Macro F1:       {macro_f1:.3f}")
print(f"Weighted F1:    {weighted_f1:.3f}")
print(f"ESI 1 recall:   {esi1_recall:.3f}")


              precision    recall  f1-score   support

           1      0.444     0.250     0.320        16
           2      0.716     0.608     0.658      3585
           3      0.660     0.758     0.706      5402
           4      0.609     0.587     0.598      1779
           5      0.482     0.111     0.181       243

    accuracy                          0.667     11025
   macro avg      0.582     0.463     0.492     11025
weighted avg      0.666     0.667     0.661     11025

Macro F1:       0.492
Weighted F1:    0.661
ESI 1 recall:   0.250


## 7. Confusion matrix

Rows are true ESI levels and columns are predicted levels. Values on the diagonal are correct predictions; off-diagonal values are errors.


In [8]:
cm = confusion_matrix(
    y_test,
    pred_logreg,
    labels=[1, 2, 3, 4, 5],
)

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[1, 2, 3, 4, 5],
).plot(
    ax=ax,
    values_format="d",
    colorbar=False,
)

ax.set_title("Logistic Regression Confusion Matrix")
ax.set_xlabel("Predicted ESI level")
ax.set_ylabel("True ESI level")
plt.tight_layout()
plt.show()


## 8. Clinical interpretation

The logistic-regression model achieved **66.7% accuracy**, compared with **37.5%** for the stratified random baseline. This shows that the dataset contains predictive signal.

However, the model identified only **4 of 16** true ESI Level 1 patients in the test set. Its ESI 1 recall was therefore **25.0%**. The main failure mode is under-triage of critically ill patients: a missed ESI 1 case could delay immediate resuscitation.

For the final Week 6 comparison, **ESI Level 1 recall** should be treated as the primary clinical safety metric. Overall accuracy, macro F1, weighted F1, and confusion matrices remain useful supporting measures.


## Interim conclusion

The logistic-regression baseline beats random guessing, but it is not clinically safe for deployment. The next step is to train a bounded decision tree, compare both models, and investigate whether defensible adjustments improve ESI Level 1 detection.
